# Editor de proyecto (anywidget)

Comprobación manual del widget: lo que no pueden verificar los tests de Python.

**Antes de empezar**: selecciona el kernel del `.venv` del proyecto (en VS Code,
arriba a la derecha → *Select Kernel* → *Python Environments* → `.venv`).

La interfaz es maestro-detalle: a la izquierda los componentes agrupados por tipo,
a la derecha el formulario del que tengas seleccionado. Los campos se generan del
esquema, así que cada uno sale con su unidad, sus límites y sus opciones.

La primera vez, el navegador descarga `ajv` de esm.sh: hace falta conexión.

## 1. Proyecto pequeño

In [1]:
import opensimula as osm

sim = osm.Simulation()
sim.console_print = False
pro = sim.new_project("proyecto")
pro.read_json("../test/test_project_1.json")

editor = pro.editor()
editor

### Qué mirar

1. Que aparezcan los dos paneles, no una caja vacía ni `Error displaying widget`.
2. Pincha un `Material`. El formulario debe traer `conductivity` como campo
   numérico con la unidad `W/(m·K)` al lado, y `use_resistance` como casilla.
3. Pon `conductivity` a `-1`: el campo se marca en rojo, con
   `must be >= 0` debajo y en la barra de estado. Si dijera *"must NOT have
   additional properties"*, el discriminador de Ajv no estaría actuando.
4. Mira un `Building_surface`: `construction` tiene que ser un **desplegable con
   los nombres de las Construction del proyecto**, no un campo de texto libre.
5. Déjalo válido antes de seguir.

## 2. ¿Llegan los cambios al kernel?

In [ ]:
# Cambia algo en el widget de arriba, espera medio segundo y ejecuta esto.
editor.value["components"][0]

In [ ]:
print("válido:", editor.is_valid())
for linea in editor.error_report():
    print(" ", linea)

Si `editor.value` no refleja lo que escribiste, la sincronización JS → Python
no funciona (consola del navegador: *Developer: Toggle Developer Tools*).

## 3. ¿Llegan los cambios del kernel al widget?

In [ ]:
# Reasignar, nunca mutar: traitlets detecta los cambios por identidad.
editor.value = {**editor.value, "description": "cambiado desde Python"}

El panel de proyecto debe mostrar el nuevo `description` sin perder lo demás.

## 4. Qué llega solo al proyecto y qué necesita Apply

No todos los cambios se comportan igual, y la barra de abajo te lo dice.

**Cambiar un valor** —un número, un desplegable, una lista de coordenadas— es
una asignación local: llega al proyecto **solo**, sin que pulses nada. El
componente sigue siendo el mismo objeto, así que una variable que apunte a él
no se queda huérfana.

**Renombrar, añadir, borrar o cambiar el `type`** obliga a reconstruir el
proyecto entero, porque los componentes se referencian por nombre y esas
referencias se resuelven al cargar. Eso espera: la barra se pone en amarillo
diciendo qué falta, y se habilita el botón **Apply**.

In [ ]:
# Un valor: llega solo
comp = pro.component_list()[0]
nombre = comp.parameter("name").value
print("antes  :", comp.parameter("float").value)

doc = {**editor.value, "components": [dict(c) for c in editor.value["components"]]}
next(c for c in doc["components"] if c["name"] == nombre)["float"] = 99.5
editor.value = doc

print("después:", pro.component(nombre).parameter("float").value, "  <- sin pulsar nada")
print("pending:", editor.pending)
print("misma instancia:", pro.component(nombre) is comp)

In [ ]:
# Algo estructural: espera
doc = {**editor.value, "components": [dict(c) for c in editor.value["components"]]}
doc["components"].append({"type": "Material", "name": "material_nuevo"})
editor.value = doc

print("pending :", editor.pending)
print("proyecto:", len(pro.component_list()), "componentes (sin tocar)")

Mira la barra del widget: debe estar en amarillo con el aviso y el botón
**Apply** activo. Púlsalo ahí, o desde el kernel:

In [ ]:
errores = editor.apply()
for e in errores:
    print(e["message"] if isinstance(e, dict) else e.text)

print("pending :", editor.pending)
print("proyecto:", len(pro.component_list()), "componentes")
print("misma instancia tras apply:", pro.component(nombre) is comp, " <- reconstruido")

`apply()` **reconstruye** el proyecto, no lo parchea. Por eso se reserva para
lo estructural:

- Las variables que apuntaran a componentes de antes quedan obsoletas; hay que
  volver a pedirlos con `pro.component(nombre)`.
- Los resultados de simulación se van con los componentes viejos.

Un valor que no cumple el esquema **no llega al proyecto**: se queda en el
documento, marcado en rojo en el formulario, y entra solo en cuanto lo
corriges.

## 5. Un edificio real

In [ ]:
hulc = sim.new_project("hulc")
hulc.read_json("edificio_curso_hulc.json")
print("componentes:", len(hulc.component_list()))

editor_hulc = hulc.editor()
editor_hulc

Aquí se ve si aguanta un edificio de verdad: 150 componentes en 15 grupos.

**Prueba el aviso de referencias colgadas**: renombra una `Construction` y mira la
barra de estado. Debe avisar en amarillo de los `Building_surface` que se quedan
apuntando a un nombre que ya no existe. Eso no lo detecta el esquema — que un
nombre exista es propiedad del documento, no del tipo — así que es una
comprobación aparte.

## 6. Marimo

El mismo widget, envuelto:

```python
import marimo as mo
import opensimula as osm

sim = osm.Simulation()
pro = sim.new_project("proyecto")
pro.read_json("test/test_project_1.json")

editor = mo.ui.anywidget(pro.editor())
editor
```

En otra celda, `editor.value["value"]` es reactivo.

## Desarrollo del JS

Para tocar `editor.js` sin reiniciar el kernel, arranca con `ANYWIDGET_HMR=1`.

In [ ]:
# Sin navegador, is_valid() y error_report() siguen funcionando:
# se calculan en Python, no dependen de lo que reporte el widget.
print("válido:", editor.is_valid())
for linea in editor.error_report():
    print(" ", linea)